# 手寫簽名辨識評估

此 Notebook 用於評估多個 LLM 模型對於 `sample` 資料夾中手寫繁體中文簽名的辨識能力。
 
評估指標包括：
1. **單字正確率 (Character Accuracy)**
2. **姓名完全正確率 (Exact Match Accuracy)**
3. **模型穩定性 (Consistency/Stability)**: 同一張圖片測試3次，結果的一致性。

In [ ]:
import os
import base64
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import Levenshtein  # 用於計算單字正確率 (編輯距離)
import httpx
import concurrent.futures
import threading

load_dotenv()

# solve SSL problem
http_client = httpx.Client(verify=False, timeout=60.0)
http_async_client = httpx.AsyncClient(verify=False)

## 1. 載入資料
讀取 `sample` 資料夾中的圖片，檔名即為正確答案（Ground Truth）。

In [ ]:
sample_dir = Path("sample")
if not sample_dir.exists():
    print("請確保 sample 資料夾存在")

# 讀取所有圖片檔案 (支援 png, jpg, jpeg)
image_files = []
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    image_files.extend(list(sample_dir.glob(ext)))

dataset = []
for img_path in image_files:
    ground_truth = img_path.stem  # 檔名即為正確答案
    dataset.append({
        "image_path": img_path,
        "ground_truth": ground_truth
    })

print(f"共找到 {len(dataset)} 張圖片。")

## 2. 模型初始化與 API 封裝

In [ ]:
import re

# --- Prompt 優化設定 (結構化與思維鏈 Chain-of-Thought) ---
SYSTEM_INSTRUCTION = "你是一個頂尖的筆跡鑑定與繁體中文手寫 OCR 專家。你的任務是精確還原圖片中的手寫簽名（通常為 2 到 4 個字的繁體中文姓名）。"

USER_PROMPT = """請仔細觀察圖片中的簽名字跡，依照以下結構化步驟進行辨識，並嚴格使用 XML 標籤輸出結果：

<thinking>
1. 筆畫拆解：逐步分析每個字的空間結構、部首、基本筆畫與草書連筆特徵。
2. 候選字推測：若字跡過於潦草，請列出視覺上形似的繁體中文字。
3. 姓名合理性審查：結合台灣常見百家姓與慣用命名用字（如：豪、傑、雅、婷、宗、翰等），從候選字中選出最符合日常邏輯的姓名組合。避免拼湊無意義的冷僻字。
</thinking>
<name>
在此處填入最終決定的純文字姓名（注意：只能有 2~4 個中文字，禁止任何標點符號、空格或額外說明）
</name>

【輸出範例】
<thinking>
第一個字上方有草字頭，下方為「早」，推測姓氏為「草」或「萬」，「萬」較符合常見姓氏。
第二個字左側為「女」，右側為「亭」，是常見字「婷」。
經過筆畫特徵與姓名邏輯比對，最合理的解答為「萬婷」。
</thinking>
<name>萬婷</name>"""

def extract_final_name(raw_text):
    """從模型的回答中提取 <name> 標籤內的內容"""
    match = re.search(r"<name>(.*?)</name>", raw_text, re.DOTALL | re.IGNORECASE)
    if match:
        # 清理可能的空白與換行
        return re.sub(r"\s+", "", match.group(1).strip())
    # 如果模型沒有照格式輸出，去除所有空白做為 fallback
    return re.sub(r"\s+", "", raw_text.strip())

# 1. Azure OpenAI (GPT-5.4)
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_version="2025-04-01-preview",
    azure_endpoint="https://project-04-openai-service.openai.azure.com/",
    api_key=os.getenv("5.4_AZURE_OPENAI_API_KEY"),
    http_client=http_client,
)

def run_azure_openai_gpt54(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    
    response = azure_openai_client.responses.create(
        model="project-04-gpt-5.4",
        instructions=SYSTEM_INSTRUCTION,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": USER_PROMPT},
                    {"type": "input_image", "image_url": f"data:image/png;base64,{encoded_string}", "detail": "high"}, 
                ],
            }
        ],
        max_output_tokens=300, # 增加 token 給 thinking 空間
    )
    return extract_final_name(response.output_text.strip())


# 2. Google Gemini
from google import genai
from google.genai import types

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"),
    http_options={'client': http_client}
)

def extract_gemini_text(response):
    candidates = getattr(response, "candidates", None) or []
    if not candidates:
        return ""
    parts = getattr(candidates[0].content, "parts", None) or []
    return "".join(part.text for part in parts if getattr(part, "text", None)).strip()

def run_gemini(model_name, image_path):
    image_bytes = image_path.read_bytes()
    image_part = types.Part.from_bytes(data=image_bytes, mime_type="image/png")
    
    combined_prompt = f"{SYSTEM_INSTRUCTION}\n\n{USER_PROMPT}"
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=[combined_prompt, image_part],
        config=types.GenerateContentConfig(
            temperature=0.2,
            max_output_tokens=300, # 增加 token 給 thinking 空間
        ),
    )
    return extract_final_name(extract_gemini_text(response))


# 3. Anthropic Azure (Claude)
from anthropic import AnthropicFoundry

claude_opus_client = AnthropicFoundry(
    api_key=os.getenv("4.5_OPUS_AZURE_ANTHROPIC_API_KEY"),
    base_url="https://project3-docai-resource.services.ai.azure.com/anthropic/",
    http_client=http_client,
)

claude_sonnet_client = AnthropicFoundry(
    api_key=os.getenv("4.6_SONNET_AZURE_ANTHROPIC_API_KEY"),
    base_url="https://9h00200-act-aifoundry.openai.azure.com/anthropic/",
    http_client=http_client,
)

def run_anthropic(client, deployment_name, image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
        
    message = client.messages.create(
        model=deployment_name,
        system=SYSTEM_INSTRUCTION,
        messages=[
            {
                "role": "user", 
                "content": [
                    {"type": "text", "text": USER_PROMPT},
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": encoded_string
                        }
                    }
                ]
            }
        ],
        max_tokens=300, # 增加 token 給 thinking 空間
        temperature=0.2,
    )
    return extract_final_name(message.content[0].text.strip())

## 3. 測試函式封裝與執行

In [ ]:
models = [
    {"name": "GPT-5.4", "func": lambda img: run_azure_openai_gpt54(img)},
    {"name": "gemini-3.1-flash-lite-preview", "func": lambda img: run_gemini("gemini-3.1-flash-lite-preview", img)},
    {"name": "gemini-3-flash-preview", "func": lambda img: run_gemini("gemini-3-flash-preview", img)},
    {"name": "gemini-3.1-pro-preview", "func": lambda img: run_gemini("gemini-3.1-pro-preview", img)},
    {"name": "claude-opus-4-5", "func": lambda img: run_anthropic(claude_opus_client, "claude-opus-4-5", img)},
    {"name": "claude-sonnet-4-6", "func": lambda img: run_anthropic(claude_sonnet_client, "project-04-claude-sonnet-4-6", img)},
]

NUM_RUNS = 3
MAX_RETRIES = 2  # 每次呼叫最多允許失敗 2 次（總共 3 次機會）
results = []
tasks = []

# 用於保護 print 輸出的鎖，避免多執行緒同時輸出排版錯亂
print_lock = threading.Lock()

def evaluate_single_task(img_path, ground_truth, model_name, func, run_idx):
    start_time = time.time()
    pred = ""
    success = False
    
    # Retry loop
    for attempt in range(MAX_RETRIES + 1):
        try:
            pred = func(img_path)
            success = True
            break  # 成功就跳出 retry 迴圈
        except Exception as e:
            with print_lock:
                print(f"[失敗] {model_name} | 圖片: {img_path.name} | 第 {run_idx} 次 (Retry {attempt}/{MAX_RETRIES}) | 錯誤: {e}")
            if attempt < MAX_RETRIES:
                # 若還有重試機會，稍微等待久一點避開 rate limit
                time.sleep(3 * (attempt + 1)) 
            else:
                pred = "" # 所有機會用盡
        
    execution_time = time.time() - start_time
    
    with print_lock:
        status_msg = "[完成]" if success else "[徹底失敗]"
        print(f"{status_msg} {model_name:<30} | 圖片: {img_path.name:<15} | 第 {run_idx} 次 | 耗時: {execution_time:.2f} 秒")
        
    return {
        "image": img_path.name,
        "ground_truth": ground_truth,
        "model": model_name,
        "run_idx": run_idx,
        "prediction": pred,
        "execution_time_sec": execution_time
    }

print("開始平行執行所有模型的評估任務...")
total_tasks = len(dataset) * len(models) * NUM_RUNS
print(f"預計執行總任務數: {total_tasks}")

# 使用 ThreadPoolExecutor 平行化執行
# MAX_WORKERS 可以依照您的網路頻寬及各 API rate limit 調整，這裡預設為 10
MAX_WORKERS = 10 

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 提交所有任務
    future_to_task = {}
    for data in dataset:
        img_path = data["image_path"]
        gt = data["ground_truth"]
        
        for model_info in models:
            model_name = model_info["name"]
            func = model_info["func"]
            
            for run_idx in range(1, NUM_RUNS + 1):
                future = executor.submit(evaluate_single_task, img_path, gt, model_name, func, run_idx)
                future_to_task[future] = model_name

    # 收集結果與進度
    completed = 0
    for future in concurrent.futures.as_completed(future_to_task):
        res = future.result()
        results.append(res)
        completed += 1
        
        # 每完成 5 個任務或是全部完成時，顯示整體進度
        if completed % 5 == 0 or completed == total_tasks:
            with print_lock:
                print(f"=== 整體進度: {completed} / {total_tasks} ({completed/total_tasks*100:.1f}%) ===")

print("所有任務執行完畢！")

## 4. 計算評估指標

利用 `Levenshtein` 計算單字正確率 (1 - 正規化編輯距離)。
利用精確比對計算姓名完全正確率。

In [ ]:
df_results = pd.DataFrame(results)

def calculate_char_accuracy(gt, pred):
    if not gt:
        return 0.0
    distance = Levenshtein.distance(gt, pred)
    max_len = max(len(gt), len(pred))
    if max_len == 0: return 1.0
    return max(0.0, 1.0 - (distance / max_len))

# 移除預測結果中的空白等可能干擾的字元
df_results['prediction_clean'] = df_results['prediction'].str.replace(r"\s+", "", regex=True)

df_results['exact_match'] = df_results['ground_truth'] == df_results['prediction_clean']
df_results['char_accuracy'] = df_results.apply(lambda row: calculate_char_accuracy(row['ground_truth'], row['prediction_clean']), axis=1)

display(df_results.head(10))

## 5. 統計與視覺化

In [ ]:
# 1. 姓名完全正確率 (Exact Match Accuracy) 依模型統計
exact_match_summary = df_results.groupby('model')['exact_match'].mean().reset_index()
exact_match_summary.rename(columns={'exact_match': 'Exact Match Accuracy'}, inplace=True)

# 2. 單字正確率 (Character Accuracy) 依模型統計
char_acc_summary = df_results.groupby('model')['char_accuracy'].mean().reset_index()
char_acc_summary.rename(columns={'char_accuracy': 'Character Accuracy'}, inplace=True)

# 3. 模型穩定性 (Consistency)
# 判斷同一張圖片的三次預測是否完全一致
def check_consistency(group):
    preds = group['prediction_clean'].tolist()
    return len(set(preds)) == 1

consistency_summary = df_results.groupby(['model', 'image']).apply(check_consistency).reset_index(name='is_consistent')
consistency_summary = consistency_summary.groupby('model')['is_consistent'].mean().reset_index()
consistency_summary.rename(columns={'is_consistent': 'Stability (Consistency)'}, inplace=True)

# 4. 平均耗時 (Average Execution Time)
time_summary = df_results.groupby('model')['execution_time_sec'].mean().reset_index()
time_summary.rename(columns={'execution_time_sec': 'Avg Runtime (sec)'}, inplace=True)

# 合併報表
final_report = pd.merge(exact_match_summary, char_acc_summary, on='model')
final_report = pd.merge(final_report, consistency_summary, on='model')
final_report = pd.merge(final_report, time_summary, on='model')

print("========== 模型評估總表 ==========")
display(final_report)

# 儲存結果
df_results.to_csv("evaluation_raw_results.csv", index=False, encoding="utf-8-sig")
final_report.to_csv("evaluation_summary.csv", index=False, encoding="utf-8-sig")
print("\n已儲存詳細結果至 evaluation_raw_results.csv，統計結果至 evaluation_summary.csv")